In [18]:
# Import required libraries
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:
# Database connection configuration to SIMPEG
DB_HOST = os.getenv('DB_HOST_SIMPEG', 'localhost')
DB_PORT = os.getenv('DB_PORT_SIMPEG', '5432')  # PostgreSQL default port
DB_NAME = os.getenv('DB_DATABASE_SIMPEG', 'your_database_name')
DB_USER = os.getenv('DB_USERNAME_SIMPEG', 'your_username')
DB_PASSWORD = os.getenv('DB_PASSWORD_SIMPEG', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connecting to database: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"User: {DB_USER}")

# Test connection
try:
    engine_simpeg = create_engine(connection_string, echo=False)
    with engine_simpeg.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")


Connecting to database: simpeg_jabar on 10.110.32.121:5432
User: postgres
✅ Database connection successful!
Connection test result: 1


In [24]:
def load_data_from_sql(query, engine):
    """
    Load data from PostgreSQL database into a pandas DataFrame.
    
    Parameters:
    query (str): SQL query to execute
    engine: SQLAlchemy engine object
    
    Returns:
    pandas.DataFrame: Data from the query
    """
    connection = None
    try:
        # Create a new connection and rollback any pending transaction
        connection = engine.connect()
        
        # Rollback any pending transaction to ensure clean state
        try:
            connection.rollback()
        except:
            pass  # If no transaction to rollback, ignore
        
        # Execute the query
        df = pd.read_sql(query, connection)
        print(f"✅ Data loaded successfully! Shape: {df.shape}")
        
        connection.close()
        return df
    except Exception as e:
        # Ensure connection is closed on error
        if connection:
            try:
                connection.rollback()
            except:
                pass
            try:
                connection.close()
            except:
                pass
        print(f"❌ Error loading data: {e}")
        return None

def get_table_info(table_name, engine):
    """
    Get basic information about a table.
    
    Parameters:
    table_name (str): Name of the table
    engine: SQLAlchemy engine object
    """
    try:
        # Get table structure using PostgreSQL information_schema
        structure_query = f"""
            SELECT 
                column_name,
                data_type,
                character_maximum_length,
                is_nullable,
                column_default
            FROM information_schema.columns
            WHERE table_name = '{table_name}'
            ORDER BY ordinal_position
        """
        structure = pd.read_sql(structure_query, engine)
        
        # Get row count
        count_query = f"SELECT COUNT(*) as row_count FROM {table_name}"
        count_result = pd.read_sql(count_query, engine)
        
        print(f"📊 Table: {table_name}")
        print(f"Rows: {count_result['row_count'].iloc[0]}")
        print(f"Columns: {len(structure)}")
        print("\nColumn Information:")
        print(structure)
        
        return structure
    except SQLAlchemyError as e:
        print(f"❌ Error getting table info: {e}")
        return None

def list_tables(engine):
    """
    List all tables in the database.
    
    Parameters:
    engine: SQLAlchemy engine object
    """
    try:
        # Use PostgreSQL information_schema to list tables
        query = """
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public'
            ORDER BY table_name
        """
        tables = pd.read_sql(query, engine)
        print("📋 Available tables:")
        for table in tables['table_name']:
            print(f"  - {table}")
        return tables
    except SQLAlchemyError as e:
        print(f"❌ Error listing tables: {e}")
        return None

print("Data loading functions defined successfully!")



Data loading functions defined successfully!


In [27]:
query = "SELECT peg_id, peg_nip, peg_nama, peg_lahir_tanggal, peg_usia, peg_jenis_kelamin, peg_status, \
peg_ketstatus, peg_umur_pensiun, peg_jabatan_tmt, peg_eselon_tmt, peg_skpd_tmt, jabatan_id, \
jabatan_jenis, jabatan_nama, eselon_id, eselon_nm, unit_kerja_id, unit_kerja_nama, \
satuan_kerja_id, satuan_kerja_nama, unit_kerja_parent_nama, kedudukan_pegawai, kedudukan_pns, \
peg_status_kepegawaian_id, peg_status_kepegawaian, tugas_tambahan_jabatan_id, tugas_tambahan_jenis, \
tugas_tambahan_jabatan_nama, tmt_tugas_tambahan, peg_jenis_pns, peg_cpns_tmt, peg_pns_tmt, \
id_status_kepegawaian, peg_lahir_tempat, peg_status_perkawinan, unit_kerja_level, unit_kerja_parent, \
jabatan_kelas, tugas_tambahan2_jabatan_id, tugas_tambahan2_jenis, tugas_tambahan2_jabatan_nama, \
tmt_tugas_tambahan2, is_gtk, nip_atasan, nama_atasan, nip_atasan_bayangan, nama_atasan_bayangan, \
unit_kerja_nama_full, is_nakes, is_atasan_bayangan, is_atasan_tugas_tambahan, atasan_kinerja \
FROM public.v_pegawai_data_akhir_tahun_2025;"

df_vpd_akhir_tahun = pd.DataFrame()

df_vpd_akhir_tahun = load_data_from_sql(query, engine_simpeg)

✅ Data loaded successfully! Shape: (68171, 53)


In [28]:
query = "SELECT peg_id, peg_nip, peg_nama, peg_lahir_tanggal, peg_usia, peg_jenis_kelamin, peg_status, \
peg_ketstatus, peg_umur_pensiun, peg_jabatan_tmt, peg_eselon_tmt, peg_skpd_tmt, jabatan_id, \
jabatan_jenis, jabatan_nama, eselon_id, eselon_nm, unit_kerja_id, unit_kerja_nama, \
satuan_kerja_id, satuan_kerja_nama, unit_kerja_parent_nama, kedudukan_pegawai, kedudukan_pns, \
peg_status_kepegawaian_id, peg_status_kepegawaian, tugas_tambahan_jabatan_id, tugas_tambahan_jenis, \
tugas_tambahan_jabatan_nama, tmt_tugas_tambahan, peg_jenis_pns, peg_cpns_tmt, peg_pns_tmt, \
id_status_kepegawaian, peg_lahir_tempat, peg_status_perkawinan, unit_kerja_level, unit_kerja_parent, \
jabatan_kelas, tugas_tambahan2_jabatan_id, tugas_tambahan2_jenis, tugas_tambahan2_jabatan_nama, \
tmt_tugas_tambahan2, is_gtk, nip_atasan, nama_atasan, nip_atasan_bayangan, nama_atasan_bayangan, \
unit_kerja_nama_full, is_nakes, is_atasan_bayangan, is_atasan_tugas_tambahan, atasan_kinerja \
FROM public.v_pegawai_data_tw4_2025;"

df_vpd_tw4 = pd.DataFrame()

df_vpd_tw4 = load_data_from_sql(query, engine_simpeg)

✅ Data loaded successfully! Shape: (68173, 53)


In [29]:
# Save as pickle
df_vpd_akhir_tahun.to_pickle(f'df_vpd_akhir_tahun.pkl')

df_vpd_tw4.to_pickle(f'df_vpd_tw4.pkl')